# Single Test Template

Copy this notebook into `tests/evals/notebooks/` and rename it for a new benchmark.

Default workflow:
- one notebook = one test
- one `MODEL_SPEC` at a time
- judge defaults centrally to `claude-opus-4-6`
- edit the cells below for task, rules, workspace, and tools


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "miso").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tests.evals import DEFAULT_RUBRIC_WEIGHTS, build_eval_case, render_single_eval_summary, run_single_model_eval


In [ ]:
TEST_ID = "my_notebook_test"
TEST_TITLE = "My Notebook Benchmark"

TASK_PROMPT = """
Inspect the workspace, complete the task, and base every claim on tool-derived evidence.

Requirements:
- Do not modify files.
- Cite relative file paths.
- Keep the final answer concise.
""".strip()

EVAL_RULES = {
    "required_paths": ["README.md"],
    "required_tool_any_of": [["read_file", "search_text", "list_directory"]],
    "min_tool_calls": 1,
    "min_final_chars": 80,
}

WORKSPACE_CONFIG = {
    "workspace_mode": "repo_copy",  # or fixture_copy
    "workspace_source": ".",         # or tests/evals/fixtures/your_fixture
}

TOOLKIT_CONFIG = {
    "allowed_toolkits": ["access_workspace_toolkit", "run_terminal_toolkit"],
    "toolkit_options": {
        "run_terminal_toolkit": {"terminal_strict_mode": True},
    },
}

CANDIDATE_INSTRUCTIONS = "Use tools deliberately. Do not guess."
RUBRIC_WEIGHTS = dict(DEFAULT_RUBRIC_WEIGHTS)


In [ ]:
MODEL_SPEC = {
    "provider": "openai",
    "model": "gpt-5",
    "label": "gpt-5",
    "api_key_env": "OPENAI_API_KEY",
}

MAX_ITERATIONS = 8
ARTIFACTS_ROOT = None


In [ ]:
EVAL_CASE = build_eval_case(
    case_id=TEST_ID,
    title=TEST_TITLE,
    task_prompt=TASK_PROMPT,
    workspace_mode=WORKSPACE_CONFIG["workspace_mode"],
    workspace_source=WORKSPACE_CONFIG["workspace_source"],
    allowed_toolkits=TOOLKIT_CONFIG["allowed_toolkits"],
    toolkit_options=TOOLKIT_CONFIG.get("toolkit_options", {}),
    rule_checks=EVAL_RULES,
    rubric_weights=RUBRIC_WEIGHTS,
    candidate_instructions=CANDIDATE_INSTRUCTIONS,
)

result = run_single_model_eval(
    repo_root=REPO_ROOT,
    case=EVAL_CASE,
    model_spec=MODEL_SPEC,
    rubric_weights=RUBRIC_WEIGHTS,
    max_iterations=MAX_ITERATIONS,
    artifacts_root=ARTIFACTS_ROOT,
)

print(render_single_eval_summary(result))


In [ ]:
result["run_artifact"]


In [ ]:
result["judge_report"]
